# $k$-Means Clustering

---

## Overview

In this lecture we begin our study of **unsupervised machine learning**. Unlike supervised learning, unsupervised learning deals with *unlabeled* data. It typically falls within:

1. **Clustering**
2. **Dimensionality Reduction**

**$k$-Means** partitions $N$ points into $k$ clusters by minimizing the **within-cluster sum of squared distances** (inertia):

$$\text{Inertia} = \sum_{j=1}^{k} \sum_{\mathbf{x} \in C_j} \|\mathbf{x} - \boldsymbol{\mu}_j\|^2$$

## Algorithm

1. Initialize $k$ random centroids $\boldsymbol{\mu}_1, \dots, \boldsymbol{\mu}_k$
2. **Assign** each point to its nearest centroid: $\text{label}(\mathbf{x}) = \arg\min_j \|\mathbf{x} - \boldsymbol{\mu}_j\|$
3. **Update** centroids: $\boldsymbol{\mu}_j = \dfrac{1}{|C_j|} \sum_{\mathbf{x} \in C_j} \mathbf{x}$
4. Repeat until centroids stop moving

---

**Dataset:** Food Nutrition (`FOOD-DATA-GROUP1.csv`) or synthetic blobs  
**Task:** Discover natural food groups.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme()

from rice_ml.unsupervised_learning import KMeans
from rice_ml.preprocess import StandardScaler

In [ ]:
try:
    df = pd.read_csv('../../../data/FOOD-DATA-GROUP1.csv')
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    X = df[num_cols].dropna().values.astype(float)
    print(f'Loaded food nutrition dataset (FOOD-DATA-GROUP1): {X.shape}')
except FileNotFoundError:
    from sklearn.datasets import make_blobs
    X, _ = make_blobs(n_samples=200, centers=4, n_features=2, random_state=0)
    print('CSV not found. using synthetic blob data')

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'Scaled shape: {X_scaled.shape}')

In [ ]:
# Visualize data (first 2 features / PCA-reduced)
from rice_ml.unsupervised_learning import PCA

X_2d = PCA(n_components=2).fit_transform(X_scaled)

plt.figure(figsize=(10, 8))
plt.scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.5, color='steelblue')
plt.xlabel('PC 1', fontsize=15)
plt.ylabel('PC 2', fontsize=15)
plt.title('Data: PCA Projection (Before Clustering)', fontsize=18)
plt.show()

## Elbow Method. Choosing $k$

Plot inertia vs $k$. The "elbow" point suggests the optimal number of clusters.

In [ ]:
inertias = []
k_range = range(1, 10)

for k in k_range:
    km = KMeans(k=k, max_iters=100)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(k_range, inertias, marker='o', color='steelblue')
plt.xlabel('Number of Clusters $k$', fontsize=15)
plt.ylabel('Inertia', fontsize=15)
plt.title('$k$-Means: Elbow Method', fontsize=18)
plt.show()

In [ ]:
k = 4
km = KMeans(k=k, max_iters=200)
km.fit(X_scaled)

print(f'Cluster sizes: {[int((km.labels_ == j).sum()) for j in range(k)]}')
print(f'Inertia: {km.inertia_:.2f}')

In [ ]:
from sklearn.metrics import silhouette_score

# Silhouette score: measures how well each point fits its cluster vs neighbours
# Range: [-1, 1]. higher is better; >0.5 is strong, 0.2–0.5 is reasonable
sil = silhouette_score(X_scaled, km.labels_)
print(f'Silhouette Score (k={k}): {sil:.4f}')

# Silhouette scores across k values
sil_scores = []
for ki in range(2, 10):
    km_i = KMeans(k=ki, max_iters=200)
    km_i.fit(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, km_i.labels_))

plt.figure(figsize=(8, 5))
plt.plot(range(2, 10), sil_scores, marker='o', color='steelblue')
plt.xlabel('k', fontsize=13)
plt.ylabel('Silhouette Score', fontsize=13)
plt.title('K-Means: Silhouette Score vs k', fontsize=15)
plt.tight_layout()
plt.show()


In [ ]:
# Visualize clusters in 2D PCA space
colors = ['red', 'lightseagreen', 'steelblue', 'magenta', 'orange', 'purple']
centroids_2d = PCA(n_components=2).fit(X_scaled).transform(km.centroids_)

plt.figure(figsize=(10, 8))
for j in range(k):
    mask = km.labels_ == j
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1],
                c=colors[j], label=f'Cluster {j}', alpha=0.6)
plt.scatter(centroids_2d[:, 0], centroids_2d[:, 1],
            c='black', marker='X', s=200, label='Centroids')
plt.xlabel('PC 1', fontsize=15)
plt.ylabel('PC 2', fontsize=15)
plt.title(f'$k$-Means (k={k}): Cluster Assignments', fontsize=18)
plt.legend(fontsize=13)
plt.show()

## Interpretation

- The **elbow** in the inertia plot marks where adding more clusters gives diminishing returns.
- Each cluster centroid represents the *prototype* of that group.
- $k$-Means assumes spherical clusters of similar size. it may struggle with irregular shapes (use DBSCAN instead).